# 02 K-Means Clustering
**Objective:** Identify EV adoption hotspots and charger demand gaps.

Team Member: Nisarg

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import duckdb

# Set style
sns.set_theme(style="whitegrid")

## 1. Feature Engineering
We will aggregate data at the Zip Code level to find adoption patterns.

In [ ]:
PARQUET_PATH = '../data/processed/Electric_Vehicle_Population_Data.parquet'

query = """
SELECT 
    "Postal Code" as zip_code,
    COUNT(*) as ev_count,
    AVG("Model Year") as avg_model_year,
    SUM(CASE WHEN "Electric Vehicle Type" = 'Battery Electric Vehicle (BEV)' THEN 1 ELSE 0 END)::FLOAT / COUNT(*) as bev_ratio,
    AVG("Electric Range") as avg_range
FROM '" + PARQUET_PATH + "'
WHERE "Postal Code" IS NOT NULL
GROUP BY zip_code
HAVING ev_count > 10
"""

cluster_df = duckdb.query(query).to_df()
cluster_df.head()

## 2. Scaling Features

In [ ]:
features = ['ev_count', 'avg_model_year', 'bev_ratio', 'avg_range']
X = cluster_df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 3. Elbow Method for Optimal K

In [ ]:
inertia = []
K_range = range(1, 11)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.plot(K_range, inertia, marker='o', color='cyan')
plt.title("Elbow Method for Optimal K")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.show()

## 4. Running K-Means (K=4)

In [ ]:
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_df['cluster'] = kmeans.fit_transform(X_scaled).argmin(axis=1) # Note: labels_ is safer
cluster_df['cluster'] = kmeans.labels_

sns.scatterplot(data=cluster_df, x='ev_count', y='bev_ratio', hue='cluster', palette='viridis', size='avg_range')
plt.title("EV Adoption Clusters by Zip Code")
plt.show()

## 5. Cluster Interpretation

In [ ]:
cluster_summary = cluster_df.groupby('cluster')[features].mean()
display(cluster_summary)

## 6. Exporting Clustered Data

In [ ]:
OUTPUT_PATH = '../data/processed/ev_with_clusters.parquet'
cluster_df.to_parquet(OUTPUT_PATH)
print(f"Clustered data saved to: {OUTPUT_PATH}")